## **MLP (Deep Learning)**

**Protocol** (same as the tabular notebooks and the LSTM): every fold trains for a fixed number of
epochs (`n_epochs`, a tuned hyperparameter) with NO early stopping and NO epoch selection, so the
validation fold is only used to score the model, once, at the end of training. Train and validation
ROC-AUC give the overfitting gap, and Optuna maximizes
`composite_score = roc_auc_mean - ALPHA * overfit_gap_roc_auc` (trials whose gap exceeds
`MAX_OVERFIT_GAP` score 0.0). The final model is trained on all dev data with the same hyperparameters
(including `n_epochs` and `batch_size`), so it follows exactly the procedure evaluated in CV.

#### **Imports and Config**

In [ ]:
import json
import os
import random 

# Suppress TF info/warning logs before importing
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import mlflow
import numpy as np
import optuna
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
import joblib
import matplotlib.pyplot as plt
import optuna.visualization.matplotlib as optuna_mpl

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Run Configuration
MODEL_TYPE = "MLP"
MOMENT = "moment_1"

N_FOLDS = 5
RANDOM_STATE = 42
N_TRIALS = 25 # Optuna trials in tuned phase

TARGET_COL = "REBOUND_TYPE" # 1 = offensive, 0 = defensive
BALANCED   = False # this file is the unbalanced version - see directory name
BASE_TAG   = "balanced" if BALANCED else "unbalanced"
SUFFIX     = f"_{BASE_TAG}"
DATA_PATH  = f"../../../../data/sequences/{MOMENT}/features/tabular/sequences_fe_tabular{SUFFIX}.csv"

GIT_COMMIT = "9eb24ae"

# Overfitting control
# Composite objective: roc_auc_mean - ALPHA * (train_mean - test_mean)
# Higher ALPHA -> stronger regularization pressure in the search.
ALPHA = 0.1
# Hard cap: trials where gap > this value are scored as 0.0 and logged
# but effectively removed from contention. Set to 1.0 to disable.
MAX_OVERFIT_GAP = 0.15

# Permutation importance (final model)
N_REPEATS_PERM = 10

# MLflow
EXP_BASELINE = f"{MODEL_TYPE}_baseline"
EXP_TUNED = f"{MODEL_TYPE}_tuned"
EXP_FINAL = f"{MODEL_TYPE}_final"
MLFLOW_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', 'mlruns'))
RUN_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}"))

os.makedirs(MLFLOW_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(MLFLOW_DIR, ".trash"), exist_ok=True)
mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR.replace(os.sep, '/')}")

print(f"Artifacts -> {RUN_DIR}")
print(f"Overfitting - alpha={ALPHA}, hard cap={MAX_OVERFIT_GAP}")
print(f"TensorFlow version: {tf.__version__}")


#### **Data Loading & Feature Selection**

In [ ]:
df = pd.read_csv(DATA_PATH)

EXCLUDED_COLS = ["GAME_ID", "EVENT_NUM", "DETECTION_CONFIDENCE",
                  "SHOT_DETECTION_CONFIDENCE", "RIM_DETECTION_CONFIDENCE"]
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL and c not in EXCLUDED_COLS]

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)
groups = df["GAME_ID"].values   # for game-level CV (StratifiedGroupKFold)

print(f"Dataset : {X.shape[0]} samples | {X.shape[1]} features | {len(np.unique(groups))} games")
print(f"Class balance - offensive: {y.mean():.3f}")

# Class-balance flag, logged alongside GIT_COMMIT via BASE_PARAMS below
class_counts = pd.Series(y).value_counts()
CLASS_BALANCE_RATIO = class_counts.min() / class_counts.max()
IS_BALANCED = bool(np.isclose(CLASS_BALANCE_RATIO, 1.0, atol=0.02))

#### **Helpers**

In [ ]:
CV = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

SCORING_KEYS = ["accuracy", "roc_auc", "f1", "precision", "recall"]

BASE_PARAMS = {
    "model_type": MODEL_TYPE,
    "moment": MOMENT,
    "git_commit": GIT_COMMIT,
    "n_features": len(FEATURE_COLS),
    "n_folds": N_FOLDS,
    "cv_strategy": "StratifiedGroupKFold",
    "random_state": RANDOM_STATE,
    "alpha": ALPHA,
    "max_overfit_gap": MAX_OVERFIT_GAP,
    "class_balanced": IS_BALANCED,
    "loss_pos_weighting": not IS_BALANCED,
}


# Set all relevant random seeds for reproducibility.
def set_seeds(seed: int = RANDOM_STATE) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# Keras class_weight dict. None on the balanced branch. On the unbalanced branch
# (IS_BALANCED is False) positives are weighted by n_neg / n_pos of the TRAINING rows,
# the equivalent of class_weight='balanced' in the tabular models.
def make_class_weight(y_train):
    if IS_BALANCED:
        return None
    n_pos = float((y_train == 1).sum())
    n_neg = float((y_train == 0).sum())
    return {0: 1.0, 1: n_neg / n_pos}


# Build and compile a dense MLP for binary classification.
#
# Architecture: [Dense -> BatchNorm -> ReLU -> Dropout] * n_layers -> Dense(1, sigmoid)
def build_model(
    n_features: int,
    n_layers: int,
    n_units: int,
    dropout_rate: float,
    learning_rate: float,
    l2: float,
) -> tf.keras.Model:
    reg = tf.keras.regularizers.l2(l2)
    inputs = tf.keras.Input(shape=(n_features,))
    x = inputs
    for _ in range(n_layers):
        x = tf.keras.layers.Dense(n_units, kernel_regularizer=reg)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Return all scoring metrics for one split.
def eval_fold(model, X_split, y_split) -> dict:
    y_prob = model.predict(X_split, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_split, y_pred),
        "roc_auc": roc_auc_score(y_split, y_prob),
        "f1": f1_score(y_split, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_split, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_split, y_pred, average="macro", zero_division=0),
    }


# Manual StratifiedGroupKFold CV for Keras models, following the same protocol as
# the tabular notebooks:
#   - each fold trains for exactly `n_epochs` epochs (no early stopping, no best-epoch
#     restore), so the validation fold never influences training;
#   - the model is then scored ONCE on the training portion and ONCE on the validation
#     portion, giving the train/validation pair the overfitting gap needs.
# Splits at the GAME_ID level (via `groups`) so no game contributes events to both the
# train and validation fold.
#
# If collect_oof=True, fold_results also carries 'oof_val_idx' and 'oof_val_probs'
# (one entry per fold), for assembling oof_predictions.csv in the final-phase cell -
# this is a no-op for baseline/tuned calls, which don't pass collect_oof.
#
# Returns (metrics_dict, fold_results_dict) where metrics_dict mirrors
# the structure produced by the sklearn-based notebooks:
#   - {metric}_mean / _std / _train_mean
#   - overfit_gap_{metric}
#   - composite_score = roc_auc_mean - alpha * overfit_gap_roc_auc
def run_cv_and_log(
    build_fn,
    batch_size: int,
    n_epochs: int,
    extra_params: dict,
    phase: str = None,
    collect_oof: bool = False,
) -> tuple:
    fold_results = {f"{split}_{m}": [] for split in ("test", "train") for m in SCORING_KEYS}
    if collect_oof:
        fold_results["oof_val_idx"] = []
        fold_results["oof_val_probs"] = []

    for fold_idx, (train_idx, val_idx) in enumerate(CV.split(X, y, groups)):
        set_seeds(RANDOM_STATE + fold_idx)  # deterministic per fold

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Per-fold scaling: fit on the TRAIN fold only, transform both (no leakage)
        scaler = MinMaxScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_val = scaler.transform(X_val)

        model = build_fn()
        model.fit(
            X_train, y_train,
            epochs=n_epochs,
            batch_size=batch_size,
            class_weight=make_class_weight(y_train),  # None on the balanced branch
            verbose=0,
        )

        for split, X_split, y_split in [("test", X_val, y_val), ("train", X_train, y_train)]:
            scores = eval_fold(model, X_split, y_split)
            for m, v in scores.items():
                fold_results[f"{split}_{m}"].append(v)

        if collect_oof:
            val_probs = model.predict(X_val, verbose=0).ravel()
            fold_results["oof_val_idx"].append(val_idx.tolist())
            fold_results["oof_val_probs"].append(val_probs.tolist())

        print(f"  fold {fold_idx + 1}: val AUC={fold_results['test_roc_auc'][-1]:.4f} | "
              f"train AUC={fold_results['train_roc_auc'][-1]:.4f} | "
              f"gap={fold_results['train_roc_auc'][-1] - fold_results['test_roc_auc'][-1]:+.4f}")

    metrics = {}
    for m in SCORING_KEYS:
        test_scores = np.array(fold_results[f"test_{m}"])
        train_scores = np.array(fold_results[f"train_{m}"])
        test_mean = float(np.mean(test_scores))
        train_mean = float(np.mean(train_scores))
        gap = train_mean - test_mean

        metrics[f"{m}_mean"] = test_mean
        metrics[f"{m}_std"] = float(np.std(test_scores))
        metrics[f"{m}_train_mean"] = train_mean
        metrics[f"overfit_gap_{m}"] = gap

    metrics["composite_score"] = (
        metrics["roc_auc_mean"] - ALPHA * metrics["overfit_gap_roc_auc"]
    )

    mlflow.log_params({**BASE_PARAMS, **extra_params})
    mlflow.log_metrics(metrics)

    if phase in ("baseline", "final"):
        # Feature list/order the saved model was trained on (read by the holdout cell)
        features_path = os.path.join(RUN_DIR, "features.json")
        with open(features_path, "w") as f:
            json.dump(FEATURE_COLS, f, indent=2)

        cv_path = os.path.join(RUN_DIR, f"cv_results_{phase}.json")
        with open(cv_path, "w") as f:
            json.dump(fold_results, f, indent=2)

    return metrics, fold_results


def print_summary(label: str, metrics: dict) -> None:
    gap = metrics["overfit_gap_roc_auc"]
    print(
        f"{label}\n"
        f"  roc_auc   : {metrics['roc_auc_mean']:.4f} +/- {metrics['roc_auc_std']:.4f}  "
        f"(train {metrics['roc_auc_train_mean']:.4f}, gap {gap:+.4f})\n"
        f"  accuracy  : {metrics['accuracy_mean']:.4f} +/- {metrics['accuracy_std']:.4f}  "
        f"(train {metrics['accuracy_train_mean']:.4f}, gap {metrics['overfit_gap_accuracy']:+.4f})\n"
        f"  composite : {metrics['composite_score']:.4f}  "
        f"[= roc_auc - {ALPHA}*gap]"
    )

#### **Baseline**

In [ ]:
# MLP Baseline architecture
BASELINE_ARCH = {
    "n_layers": 3,
    "n_units": 128,
    "dropout_rate": 0.3,
    "learning_rate": 1e-3,
    "l2": 1e-4,
    "batch_size": 64,
    "n_epochs": 30,
}

def baseline_build_fn():
    return build_model(
        n_features=X.shape[1],
        n_layers=BASELINE_ARCH["n_layers"],
        n_units=BASELINE_ARCH["n_units"],
        dropout_rate=BASELINE_ARCH["dropout_rate"],
        learning_rate=BASELINE_ARCH["learning_rate"],
        l2=BASELINE_ARCH["l2"],
    )

mlflow.set_experiment(EXP_BASELINE)

with mlflow.start_run(run_name=MODEL_TYPE):
    metrics, _ = run_cv_and_log(
        build_fn=baseline_build_fn,
        batch_size=BASELINE_ARCH["batch_size"],
        n_epochs=BASELINE_ARCH["n_epochs"],
        extra_params={**BASELINE_ARCH, "phase": "baseline"},
        phase="baseline",
    )

print_summary("Baseline", metrics)

#### **Tuned (Optuna)**

**Objective**: `composite_score = roc_auc_mean - alpha * overfit_gap`  
Trials where `overfit_gap_roc_auc > MAX_OVERFIT_GAP` are penalized to 0.0.

Tuning covers the architecture (layers, units, dropout), training (learning rate, L2, batch size) and the
epoch budget `n_epochs`, which takes the place of early stopping.

> **Note** - each trial runs a full 5-fold CV with Keras training. This is slower than tree models.  
> Reduce `N_TRIALS` or the `n_epochs` range if needed.

In [ ]:
mlflow.set_experiment(EXP_TUNED)

def progress_callback(study, trial):
    print(f"Trial {trial.number + 1}/{N_TRIALS} - composite: {trial.value:.4f} | best so far: {study.best_value:.4f}")


def objective(trial):
    arch = {
        "n_layers": trial.suggest_int("n_layers", 1, 4),
        "n_units": trial.suggest_categorical("n_units", [32, 64, 128, 256, 512]),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.0, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "l2": trial.suggest_float("l2", 1e-5, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        "n_epochs": trial.suggest_int("n_epochs", 10, 100, step=10),
    }

    def trial_build_fn():
        return build_model(
            n_features=X.shape[1],
            n_layers=arch["n_layers"],
            n_units=arch["n_units"],
            dropout_rate=arch["dropout_rate"],
            learning_rate=arch["learning_rate"],
            l2=arch["l2"],
        )

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.set_tag("optuna_trial", trial.number)
        metrics, _ = run_cv_and_log(
            build_fn=trial_build_fn,
            batch_size=arch["batch_size"],
            n_epochs=arch["n_epochs"],
            extra_params={**arch, "phase": "tuned"},
        )

        gap = metrics["overfit_gap_roc_auc"]

        if gap > MAX_OVERFIT_GAP:
            mlflow.log_metric("pruned_overfit", 1.0)
            return 0.0

        mlflow.log_metric("pruned_overfit", 0.0)
        return metrics["composite_score"]


study = optuna.create_study(
    direction="maximize",
    study_name=f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, callbacks=[progress_callback])

print(f"\nBest trial - composite_score: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Inspect overfitting
trials_df = study.trials_dataframe()

valid = trials_df[trials_df["value"] > 0].copy()

print(f"Valid trials (gap <= {MAX_OVERFIT_GAP}): {len(valid)} / {len(trials_df)}")
print(valid[["value"]].describe().T.to_string())

assert len(valid) > 0, "Every trial exceeded MAX_OVERFIT_GAP - best_params would come from a discarded trial. Loosen the search space or the cap."

In [ ]:
mlflow.set_experiment(EXP_TUNED)

with mlflow.start_run(run_name=f"best_{MODEL_TYPE}"):
    mlflow.log_params({
        **BASE_PARAMS,
        **study.best_params,
        "phase": "tuned_best",
        "source": "optuna_best",
    })
    mlflow.log_metric("composite_score", study.best_value)

    optuna_mpl.plot_param_importances(study)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, "optuna_param_importance.png"), dpi=150)
    plt.close()

    optuna_mpl.plot_contour(study, params=["n_layers", "dropout_rate"])
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, "optuna_contour_layers_dropout.png"), dpi=150)
    plt.close()

#### **Final**

In [ ]:
FINAL_ARCH = dict(study.best_params)   # includes n_epochs and batch_size; used unchanged below

def final_build_fn():
    return build_model(
        n_features=X.shape[1],
        n_layers=FINAL_ARCH["n_layers"],
        n_units=FINAL_ARCH["n_units"],
        dropout_rate=FINAL_ARCH["dropout_rate"],
        learning_rate=FINAL_ARCH["learning_rate"],
        l2=FINAL_ARCH["l2"],
    )

mlflow.set_experiment(EXP_FINAL)

with mlflow.start_run(run_name=MODEL_TYPE) as final_run:
    metrics, fold_results = run_cv_and_log(
        build_fn=final_build_fn,
        batch_size=FINAL_ARCH["batch_size"],
        n_epochs=FINAL_ARCH["n_epochs"],
        extra_params={**FINAL_ARCH, "phase": "final"},
        phase="final",
        collect_oof=True,
    )

    # Refit on all dev data with the same n_epochs and batch_size as the CV folds
    set_seeds(RANDOM_STATE)
    final_scaler = MinMaxScaler().fit(X)
    X_scaled = final_scaler.transform(X)
    final_model = final_build_fn()
    final_model.fit(
        X_scaled, y,
        epochs=FINAL_ARCH["n_epochs"],
        batch_size=FINAL_ARCH["batch_size"],
        class_weight=make_class_weight(y),  # None on the balanced branch
        verbose=0,
    )

    model_path = os.path.join(RUN_DIR, "final_model.keras")
    final_model.save(model_path)
    mlflow.log_artifact(model_path)
    # The saved network expects MinMax-scaled inputs, so persist the scaler with it.
    scaler_path = os.path.join(RUN_DIR, "final_scaler.joblib")
    joblib.dump(final_scaler, scaler_path)
    mlflow.log_artifact(scaler_path)

FINAL_RUN_ID = final_run.info.run_id
print_summary("Final", metrics)

In [ ]:
# Out-of-fold probabilities, used by calibrate.py
oof_val_idx = np.concatenate([np.array(idx) for idx in fold_results["oof_val_idx"]])
oof_val_probs = np.concatenate([np.array(p) for p in fold_results["oof_val_probs"]])

oof_probs = np.empty(len(y))
oof_probs[oof_val_idx] = oof_val_probs  # val_idx across folds covers every row exactly once

oof_df = pd.DataFrame({
    "GAME_ID": df["GAME_ID"].values,
    "EVENT_NUM": df["EVENT_NUM"].values,
    "y_true": y,
    "y_prob": oof_probs,
})

oof_path = os.path.join(RUN_DIR, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(oof_path)

print(f"OOF predictions: {len(oof_df)} rows | true offensive rate {oof_df['y_true'].mean():.3f} | mean predicted prob {oof_df['y_prob'].mean():.3f}")

#### **Feature Importance (Final Model)**

MLPs have no native feature importances. Permutation importance is computed on the  
last CV fold's validation split: each feature is shuffled independently and the drop  
in ROC-AUC is recorded as its importance.

In [ ]:
# Use last fold's validation split
train_idx, val_idx = list(CV.split(X, y, groups))[-1]
X_val, y_val = X[val_idx], y[val_idx]
# final_model expects MinMax-scaled inputs; permuting scaled columns is equivalent to
# permuting raw ones (scaling is per-column monotonic), so scale once here.
X_val = final_scaler.transform(X_val)

baseline_auc = roc_auc_score(
    y_val, final_model.predict(X_val, verbose=0).ravel()
)

perm_scores = np.zeros((X.shape[1], N_REPEATS_PERM))

for i in range(X.shape[1]):
    for r in range(N_REPEATS_PERM):
        rng = np.random.default_rng(RANDOM_STATE + r)
        X_perm = X_val.copy()
        X_perm[:, i] = rng.permutation(X_perm[:, i])
        perm_auc = roc_auc_score(
            y_val, final_model.predict(X_perm, verbose=0).ravel()
        )
        perm_scores[i, r] = baseline_auc - perm_auc

importances = pd.Series(perm_scores.mean(axis=1), index=FEATURE_COLS)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot.barh(ax=ax)
ax.set_title("MLP Feature Importances - Top 20 (Permutation)")
ax.set_xlabel("Mean decrease in ROC-AUC")
plt.tight_layout()

fig_path = os.path.join(RUN_DIR, "feature_importances.png")
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved -> {fig_path}")

#### **Holdout Evaluation**

In [ ]:
# Holdout counterpart of DATA_PATH (holdout is never balanced)
HOLDOUT_DATA_PATH = f"../../../../data/sequences/{MOMENT}/features/tabular/sequences_fe_tabular_holdout.csv"

df_holdout = pd.read_csv(HOLDOUT_DATA_PATH)

with open(os.path.join(RUN_DIR, "features.json")) as f:
    holdout_feature_cols = json.load(f)

missing = set(holdout_feature_cols) - set(df_holdout.columns)
assert not missing, f"Holdout is missing features used in training: {missing}"

X_holdout = df_holdout[holdout_feature_cols].values.astype(np.float32)
y_holdout = df_holdout[TARGET_COL].values.astype(np.float32)

# final_scaler was fit on ALL dev rows (not per-fold) - reuse it as-is, never refit.
X_holdout_scaled = final_scaler.transform(X_holdout)

y_prob = final_model.predict(X_holdout_scaled, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

holdout_metrics = {
    "accuracy": accuracy_score(y_holdout, y_pred),
    "roc_auc": roc_auc_score(y_holdout, y_prob),
    "f1": f1_score(y_holdout, y_pred, average="macro"),
    "precision": precision_score(y_holdout, y_pred, average="macro"),
    "recall": recall_score(y_holdout, y_pred, average="macro"),
}

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_metrics({f"holdout_{k}": v for k, v in holdout_metrics.items()})
    mlflow.log_param("n_holdout", len(df_holdout))

print(f"Holdout: {len(df_holdout)} rows, offensive rate {y_holdout.mean():.3f}")
for k, v in holdout_metrics.items():
    print(f"  holdout_{k:9s}: {v:.4f}")